# Iris Classification - Model Training and Comparison

This notebook demonstrates training multiple machine learning models and comparing their performance.

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from iris_classifier import IrisDataLoader, ModelFactory, ModelEvaluator, IrisVisualizer
from iris_classifier.models import ModelTrainer

%matplotlib inline

## 1. Load and Split Data

In [ ]:
# Load data
data_loader = IrisDataLoader()
X_train, X_test, y_train, y_test = data_loader.get_train_test_split(test_size=0.3, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

## 2. Available Models

In [ ]:
# List available models
available_models = ModelFactory.list_available_models()

print("Available Models:")
for name, description in available_models.items():
    print(f"  - {name:20s}: {description}")

## 3. Train a Single Model

In [ ]:
# Train Decision Tree
model = ModelFactory.create_model('decision_tree')
trainer = ModelTrainer(model, 'decision_tree')
trainer.train(X_train, y_train)

print("Decision Tree model trained successfully!")

## 4. Evaluate the Model

In [ ]:
# Evaluate
evaluator = ModelEvaluator()
results = evaluator.evaluate_model(model, X_test, y_test, 'decision_tree')

evaluator.print_evaluation_report(results)

## 5. Visualize Results

In [ ]:
# Plot confusion matrix
visualizer = IrisVisualizer()
visualizer.plot_confusion_matrix(y_test, results['predictions'], model_name='Decision Tree')

## 6. Compare All Models

In [ ]:
# Get all models
models = ModelFactory.get_all_models()

# Compare
comparison_df = evaluator.compare_models(models, X_train, y_train, X_test, y_test, cv=5)

print("\nModel Comparison Results:")
display(comparison_df)

In [ ]:
# Visualize comparison
visualizer.plot_model_comparison(comparison_df, metric='Accuracy')

## 7. Identify Best Model

In [ ]:
# Get best model
best_model_name = evaluator.get_best_model(comparison_df, 'Accuracy')
best_accuracy = comparison_df[comparison_df['Model'] == best_model_name]['Accuracy'].values[0]

print(f"Best Model: {best_model_name}")
print(f"Accuracy: {best_accuracy:.4f}")

## 8. Train and Evaluate Best Model

In [ ]:
# Train best model
best_model = ModelFactory.create_model(best_model_name)
best_model.fit(X_train, y_train)

# Detailed evaluation
best_results = evaluator.evaluate_model(best_model, X_test, y_test, best_model_name)
evaluator.print_evaluation_report(best_results)

In [ ]:
# Plot confusion matrix for best model
visualizer.plot_confusion_matrix(y_test, best_results['predictions'], model_name=best_model_name)

## 9. Make Predictions on New Data

In [ ]:
# Create a new sample
new_sample = data_loader.predict_sample(5.0, 3.6, 1.4, 0.2)

# Predict
prediction = best_model.predict(new_sample)[0]
probabilities = best_model.predict_proba(new_sample)[0] if hasattr(best_model, 'predict_proba') else None

print("New Sample Prediction:")
print(f"  Input: Sepal Length=5.0, Sepal Width=3.6, Petal Length=1.4, Petal Width=0.2")
print(f"  Predicted Species: {data_loader.target_names[prediction]}")

if probabilities is not None:
    print("\n  Probabilities:")
    for i, (name, prob) in enumerate(zip(data_loader.target_names, probabilities)):
        print(f"    {name:12s}: {prob:6.2%}")

## 10. Feature Importance (for tree-based models)

In [ ]:
# Train a Random Forest for feature importance
rf_model = ModelFactory.create_model('random_forest')
rf_model.fit(X_train, y_train)

# Plot feature importance
visualizer.plot_feature_importance(rf_model)

## Key Findings

1. Multiple machine learning algorithms can achieve high accuracy on the Iris dataset
2. The best performing model can be identified through systematic comparison
3. Petal features (length and width) are typically the most important for classification
4. The dataset is relatively easy to classify due to clear separability between species